| Model            | Input per timestep                          |
| ---------------- | ------------------------------------------- |
| Baseline         | Word2Vec word vector only                   |
| Melody Variant 1 | Word2Vec + global MIDI feature vector       |
| Melody Variant 2 | Word2Vec + time-aligned MIDI feature vector |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.nn.functional as F
MAX_WORDS = 120
WORDS_PER_LINE = 8
END_TOKEN = "<EOS>"


In [2]:
# -------------------------
# 1. Reproducibility + Device
# -------------------------

import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
# -------------------------
# 2. Special Tokens
# -------------------------

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

word2idx = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
idx2word = {idx: token for token, idx in word2idx.items()}

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
SOS_IDX = word2idx[SOS_TOKEN]
EOS_IDX = word2idx[EOS_TOKEN]

print(word2idx)

{'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}


In [ ]:
# -------------------------
# 3. Model Hyperparameters
# -------------------------

WORD_DIM = 300
GLOBAL_MELODY_DIM = 7
ALIGNED_MELODY_DIM = 4

HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.2

BATCH_SIZE = 16
SEQ_LEN = 30
LEARNING_RATE = 1e-3
EPOCHS = 5

# LSTM model

In [6]:
# -------------------------
# 4. LSTM Model
# -------------------------

class LyricsLSTM(nn.Module):
    def __init__(
        self,
        word_dim,
        melody_dim,
        hidden_dim,
        vocab_size,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.word_dim = word_dim
        self.melody_dim = melody_dim

        self.lstm = nn.LSTM(
            input_size=word_dim + melody_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, word_vectors, melody_features=None):
        if self.melody_dim > 0:
            if melody_features is None:
                raise ValueError("melody_features is required for this model")
            x = torch.cat([word_vectors, melody_features], dim=-1)
        else:
            x = word_vectors

        out, _ = self.lstm(x)
        out = self.dropout(out)
        logits = self.fc(out)

        return logits

#  Load Lyrics CSV


In [7]:
import pandas as pd

CSV_PATH = r"D:\Masters Study\2ndyear\Deep_Learning\DL-Assignment-3\Data\Archive (1)\lyrics_train_set.csv"

df = pd.read_csv(CSV_PATH, header=None)

df = df.iloc[:, :3]
df.columns = ["artist", "title", "lyrics"]

print(df.head())
print(df.columns)
print(df.shape)

           artist                           title  \
0      elton john              candle in the wind   
1  gerry rafferty                    baker street   
2  gerry rafferty             right down the line   
3     2 unlimited                    tribal dance   
4     2 unlimited  let the beat control your body   

                                              lyrics  
0  goodbye norma jean & though i never knew you a...  
1  winding your way down on baker street & lite i...  
2  you know i need your love & you've got that ho...  
3  come on check it out ya'll & (come on come on!...  
4  let the beat control your body & let the beat ...  
Index(['artist', 'title', 'lyrics'], dtype='object')
(600, 3)


In [9]:
# -------------------------
# 7. Text Cleaning
# -------------------------

import re

def clean_text(text):
    text = text.lower()

    # replace & with newline marker
    text = text.replace("&", " <NEWLINE> ")

    text = re.sub(r"[^a-zA-Z0-9\s'<NEWLINE>]", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
df["clean_lyrics"] = df["lyrics"].apply(clean_text)

print(df["clean_lyrics"].iloc[0][:500])

goodbye norma jean <NEWLINE> though i never knew you at all <NEWLINE> you had the grace to hold yourself <NEWLINE> while those around you crawled <NEWLINE> they crawled out of the woodwork <NEWLINE> and they whispered into your brain <NEWLINE> they set you on the treadmill <NEWLINE> and they made you change your name <NEWLINE> and it seems to me you lived your life <NEWLINE> like a candle in the wind <NEWLINE> never knowing who to cling to <NEWLINE> when the rain set in <NEWLINE> and i would lik


In [12]:
# -------------------------
# 8. Tokenization
# -------------------------

def tokenize(text):
    return text.split()

df["tokens"] = df["clean_lyrics"].apply(tokenize)

print(df["tokens"].iloc[0][:30])

['goodbye', 'norma', 'jean', '<NEWLINE>', 'though', 'i', 'never', 'knew', 'you', 'at', 'all', '<NEWLINE>', 'you', 'had', 'the', 'grace', 'to', 'hold', 'yourself', '<NEWLINE>', 'while', 'those', 'around', 'you', 'crawled', '<NEWLINE>', 'they', 'crawled', 'out', 'of']


In [13]:
# -------------------------
# 9. Build Vocabulary
# -------------------------

from collections import Counter

word_counter = Counter()

for tokens in df["tokens"]:
    word_counter.update(tokens)

MIN_FREQ = 2

vocab_words = [
    word for word, count in word_counter.items()
    if count >= MIN_FREQ
]

print("Vocabulary size before filtering:", len(word_counter))
print("Vocabulary size after filtering:", len(vocab_words))

Vocabulary size before filtering: 7702
Vocabulary size after filtering: 4149


In [15]:
for word in vocab_words:
    if word not in word2idx:
        idx = len(word2idx)
        word2idx[word] = idx
        idx2word[idx] = word

vocab_size = len(word2idx)

print("Final vocab size:", vocab_size)

Final vocab size: 4153


In [16]:
# -------------------------
# 10. Convert Words to IDs
# -------------------------

def tokens_to_ids(tokens):
    ids = []

    for token in tokens:
        ids.append(word2idx.get(token, UNK_IDX))

    return ids

df["token_ids"] = df["tokens"].apply(tokens_to_ids)

print(df["token_ids"].iloc[0][:30])

[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 7, 12, 15, 16, 17, 18, 19, 20, 7, 21, 22, 23, 12, 24, 7, 25, 24, 26, 27]


In [17]:
import gensim.downloader as api

word2vec = api.load("glove-wiki-gigaword-300")

In [18]:
# -------------------------
# 11. Load GloVe Embeddings
# -------------------------

import gensim.downloader as api

word2vec = api.load("glove-wiki-gigaword-300")

print("Embeddings loaded")

Embeddings loaded


In [19]:
# -------------------------
# 12. Build Embedding Matrix
# -------------------------

embedding_matrix = np.zeros(
    (vocab_size, WORD_DIM),
    dtype=np.float32
)

missing_words = 0

for word, idx in word2idx.items():

    if word in word2vec:
        embedding_matrix[idx] = word2vec[word]

    else:
        embedding_matrix[idx] = np.random.normal(
            0,
            0.1,
            WORD_DIM
        )

        missing_words += 1

embedding_matrix[PAD_IDX] = np.zeros(WORD_DIM)

embedding_matrix = torch.tensor(
    embedding_matrix,
    dtype=torch.float32
).to(DEVICE)

print("Embedding matrix shape:", embedding_matrix.shape)
print("Missing words:", missing_words)

Embedding matrix shape: torch.Size([4153, 300])
Missing words: 392


In [20]:
# -------------------------
# 13. Test Embeddings
# -------------------------

sample_ids = torch.tensor(
    df["token_ids"].iloc[0][:10],
    dtype=torch.long
).to(DEVICE)

sample_vectors = embedding_matrix[sample_ids]

print(sample_vectors.shape)

torch.Size([10, 300])


In [21]:
# -------------------------
# 14. Train / Validation Split
# -------------------------

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 540
Validation size: 60


# 15. Sequence Preparation


In [23]:
# -------------------------
# 15. Sequence Preparation
# -------------------------

def create_input_target_sequences(token_ids, seq_len):

    input_sequences = []
    target_sequences = []

    for i in range(len(token_ids) - seq_len):

        input_seq = token_ids[i:i + seq_len]
        target_seq = token_ids[i + 1:i + seq_len + 1]

        input_sequences.append(input_seq)
        target_sequences.append(target_seq)

    return input_sequences, target_sequences

# 16. Dataset Class


In [24]:
# -------------------------
# 16. Dataset Class
# -------------------------

from torch.utils.data import Dataset

class LyricsDataset(Dataset):

    def __init__(self, dataframe, seq_len):

        self.seq_len = seq_len

        self.samples = []

        for _, row in dataframe.iterrows():

            token_ids = row["token_ids"]

            if len(token_ids) <= seq_len:
                continue

            input_seqs, target_seqs = create_input_target_sequences(
                token_ids,
                seq_len
            )

            for inp, tgt in zip(input_seqs, target_seqs):

                self.samples.append((inp, tgt))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        input_ids, target_ids = self.samples[idx]

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        target_ids = torch.tensor(target_ids, dtype=torch.long)

        return input_ids, target_ids

# 17. Create Dataset Objects


In [25]:
# -------------------------
# 17. Create Dataset Objects
# -------------------------

train_dataset = LyricsDataset(
    train_df,
    SEQ_LEN
)

val_dataset = LyricsDataset(
    val_df,
    SEQ_LEN
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 144054
Validation samples: 17617


# 18. DataLoaders


In [26]:
# -------------------------
# 18. DataLoaders
# -------------------------

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# 19. Test DataLoader


In [27]:
# -------------------------
# 19. Test DataLoader
# -------------------------

input_ids, target_ids = next(iter(train_loader))

print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)

Input shape: torch.Size([16, 30])
Target shape: torch.Size([16, 30])


# 20. Baseline Model


In [28]:
# -------------------------
# 20. Baseline Model
# -------------------------

baseline_model = LyricsLSTM(
    word_dim=WORD_DIM,
    melody_dim=0,
    hidden_dim=HIDDEN_DIM,
    vocab_size=vocab_size,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

# 21. Training Functions


In [29]:
# -------------------------
# 21. Training Functions
# -------------------------

def train_one_epoch(model, data_loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for input_ids, target_ids in data_loader:
        input_ids = input_ids.to(DEVICE)
        target_ids = target_ids.to(DEVICE)

        word_vectors = embedding_matrix[input_ids]

        optimizer.zero_grad()

        logits = model(word_vectors)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            target_ids.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
     
        total_loss += loss.item()

    return total_loss / len(data_loader)


def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for input_ids, target_ids in data_loader:
            input_ids = input_ids.to(DEVICE)
            target_ids = target_ids.to(DEVICE)

            word_vectors = embedding_matrix[input_ids]

            logits = model(word_vectors)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                target_ids.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(data_loader)

# 22. Train Baseline


In [30]:
# -------------------------
# 22. Train Baseline
# -------------------------

baseline_train_losses = []
baseline_val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        baseline_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss = evaluate(
        baseline_model,
        val_loader,
        criterion
    )

    baseline_train_losses.append(train_loss)
    baseline_val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 1/5 | Train Loss: 4.0066 | Val Loss: 5.0569
Epoch 2/5 | Train Loss: 2.7444 | Val Loss: 5.4043
Epoch 3/5 | Train Loss: 2.3238 | Val Loss: 5.6146
Epoch 4/5 | Train Loss: 2.1052 | Val Loss: 5.8214
Epoch 5/5 | Train Loss: 1.9708 | Val Loss: 5.9019
